In [ ]:

# class Model(nn.Module):
#     def __init__(self):
#         super().__init__()
#         emb_size = 10
#         hidden_size = 128
#         n_layers = 6

#         self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
#         self.conv2 = nn.Conv2d(16, 16, 3, padding=1)
#         self.conv3 = nn.Conv2d(16, 1, 3, padding=1)
#         self.fc1 = nn.Linear(100, hidden_size)
#         self.fcx = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_layers)])
#         self.fc2 = nn.Linear(hidden_size, 40*40*4)
#         self.LayerNorm = nn.LayerNorm(100)
#         self.LayerNormFC = nn.LayerNorm(hidden_size)
    
#     def forward(self, inputs):
#         c1 = F.gelu(self.conv1(inputs))
#         p1 = F.max_pool2d(c1, 2)
#         c2 = F.gelu(self.conv2(p1))
#         p2 = F.max_pool2d(c2, 2)
#         c3 = F.gelu(self.conv3(p2))
#         c3 = self.LayerNorm(c3.view(c3.shape[0], -1))
#         x = F.gelu(self.fc1(c3))
#         for fc in self.fcx:
#             x = self.LayerNormFC(x + F.gelu(fc(x)))
#         return self.fc2(x)



In [33]:
import numpy as np
import tqdm

# Notes
# - Enforce lazyness
# - Enforce types
# - Make tests before features

# TODOS
# - Implement backpropagation
# - Implement OPs in another file
# - Use AMD gpu for ops

ids = 0

class ShapeTracker:
    def __init__(self, shape) -> None:
        self.shape = shape
        # Handle Reshape, Transpose, permute, pad, stride, unpad

    def view(self, *args):
        return ShapeTracker(args)
    

class Tensor():
    def __init__(self, value, requires_grad=False) -> None:
        self.realized = True
        self.value = value
        #self.shape = value.shape
        self.id = None
        self.shape = ShapeTracker(value.shape)
        self.requires_grad = requires_grad

    def realize(self): return self
    def numpy(self): return self.value if self.realized else self.realize().value

    def __add__(self, other): return LazyTracker(self, other, np.add)
    def __sub__(self, other): return LazyTracker(self, other, np.subtract)
    def __mul__(self, other): return LazyTracker(self, other, np.multiply)
    def __rmul__(self, other): return self.__mul__(other)

    def __truediv__(self, other): return LazyTracker(self, other, np.divide)
    def __pow__(self, other): return LazyTracker(self, other, np.power)
    def __matmul__(self, other): return LazyTracker(self, other, np.matmul)
    def __neg__(self): return LazyTracker(self, self, np.negative)
    def __abs__(self): return LazyTracker(self, self, np.abs)

    def __str__(self) -> str: return f"Tensor: {self.value}" if self.realized else "(LazyTracker: " + str(self.a) + "  " +  str(self.b) + " " + str(self.f) + ")"
            
    def tanh(self): return LazyTracker(self, self, np.tanh)
    def exp(self): return LazyTracker(self, self, np.exp)

    def get_graph(self, graph=[]):
        global ids
        try:
            if self.id is None:
                self.id = ids
                ids+=1
            if (isinstance(self, LazyTracker)):
                self.a.get_graph(graph)
                self.b.get_graph(graph)
                graph.append([self.id, self.a.id, self.f, self.b.id])   
            return graph
        except:
            graph.append(None)
            return graph

OPs = {
    "Movement": ["Reshape", "Transpose", "Permute", "Pad", "Stride", "Unpad"],
    "Unary": ["Abs", "Neg", "Sqrt"],
    "Binary": ["Add", "Sub", "Mul", "Div", "Pow", "Matmul"],
    "Ternary": ["Conv2d", "Conv3d", "Conv1d"],
    "Reduce": ["Sum", "Mean", "Max", "Min"],
}

class LazyTracker(Tensor):
    def __init__(self, a, b, f) -> None:
        self.a = a
        self.b = b
        self.f = f
        self.realized = False
        self.id = None

    def realize(self):
        self.value = self.f(self.a.realize().value, self.b.realize().value)
        self.realized = True
        return self

    def simplify(self):
        graph = self.get_graph()
        return graph


class Linear:
    def __init__(self, in_size, out_size, init=np.random.randn) -> None:
        self.weights = Tensor(init(in_size, out_size))
        self.bias = Tensor(init(out_size))
    
    def __call__(self, x):
        return x @ self.weights + self.bias
    
class Conv2d:
    def __init__(self) -> None:
        pass

class LayerNorm:
    def __init__(self, shape) -> None:
        pass

class ModuleList:
    def __init__(self, layers) -> None:
        pass

def tanh(x):
    return x.exp() - (-x).exp() / (x.exp() + (-x).exp())

def gelu(x):
    return x * (np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * np.power(x, 3))) + 1) * 0.5

In [35]:
c = Tensor(np.array([1,2,3]))
[elm for elm in gelu(c).simplify() if elm != None]

[[6, 2, <ufunc 'add'>, 7],
 [6, 2, <ufunc 'add'>, 7],
 [4, 5, <ufunc 'tanh'>, 5],
 [1, 2, <ufunc 'multiply'>, 3],
 [15, 11, <ufunc 'add'>, 16],
 [15, 11, <ufunc 'add'>, 16],
 [13, 14, <ufunc 'tanh'>, 14],
 [10, 11, <ufunc 'multiply'>, 12]]

In [ ]:

class Model(Module):
    def __init__(self):
        super().__init__()
        emb_size = 10
        hidden_size = 128
        n_layers = 6

        self.conv1 = Conv2d(3, 16, 3, padding=1)
        self.conv2 = Conv2d(16, 16, 3, padding=1)
        self.conv3 = Conv2d(16, 1, 3, padding=1)
        self.fc1 = Linear(100, hidden_size)
        self.fcx = ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(n_layers)])
        self.fc2 = Linear(hidden_size, 40*40*4)
        self.LayerNorm = LayerNorm(100)
        self.LayerNormFC = LayerNorm(hidden_size)
    
    def forward(self, inputs):
        c1 = gelu(self.conv1(inputs))
        p1 = max_pool2d(c1, 2)
        c2 = gelu(self.conv2(p1))
        p2 = max_pool2d(c2, 2)
        c3 = gelu(self.conv3(p2))
        c3 = self.LayerNorm(c3.view(c3.shape[0], -1))
        x = gelu(self.fc1(c3))
        for fc in self.fcx:
            x = self.LayerNormFC(x + gelu(fc(x)))
        return self.fc2(x)

In [2]:
a = Tensor(np.array([1, 2, 3, 4, 5]), requires_grad=True)
b = Tensor(np.array([2, 2, 2, 2, 2]), requires_grad=True)
k = Tensor(np.array([1, 1, 1, 1, 1]), requires_grad=True)
c = a + b + a + k
d = c @ a

In [4]:
d.numpy()

155

In [73]:
l1 = Linear(5, 1)
e = l1(a)

In [74]:
print(e)

(LazyTracker: (LazyTracker: Tensor: [1 2 3 4 5]  Tensor: [[ 2.17595495]
 [-0.1544714 ]
 [ 0.1308235 ]
 [ 0.72305676]
 [-0.47084311]] <ufunc 'matmul'>)  Tensor: [-0.53857683] <ufunc 'add'>)


In [80]:
print(e.numpy())

[2.25891732]


In [31]:
print(d)

Tensor: 155


In [30]:
print(c)
print(d.numpy())

(LazyTracker: (LazyTracker: (LazyTracker: Tensor: [1 2 3 4 5]  Tensor: [2 2 2 2 2] <ufunc 'add'>)  Tensor: [1 2 3 4 5] <ufunc 'add'>)  Tensor: [1 1 1 1 1] <ufunc 'add'>)
155


In [22]:
s = c.simplify()
print("".join([str(a) + "\n" for a in s]))


[2, 3, <ufunc 'add'>, 4]
[1, 2, <ufunc 'add'>, 3]
[0, 1, <ufunc 'add'>, 5]



In [5]:
print(c)
print(c.numpy())


(LazyTracker: (LazyTracker: (LazyTracker: Tensor: [1 2 3 4 5]  Tensor: [2 2 2 2 2] <ufunc 'add'>)  Tensor: [1 2 3 4 5] <ufunc 'add'>)  Tensor: [1 1 1 1 1] <ufunc 'add'>)
[ 5  7  9 11 13]


In [6]:
d.realize()

In [7]:
d.numpy()

array([ 6,  9, 12, 15, 18])

In [8]:
c = a + b + a
d = (a*b) - (a**b)
e = c+d

In [9]:
g = c.numpy()
print(g[0])

4


In [1]:
import torch

In [4]:
c = torch.zeros((25600, 10000))